In [ ]:
!pip -q install tensorflow tensorflow-hub librosa pandas

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import numpy as np
import pandas as pd
from google.colab import files

print("TensorFlow:", tf.__version__)

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

# YAMNet's class names
class_map_path = yamnet.class_map_path().numpy().decode("utf-8")
class_names = pd.read_csv(class_map_path)["display_name"].tolist()

print("YAMNet loaded")
print("Number of classes:", len(class_names))

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow: 2.20.0
YAMNet loaded
Number of classes: 521


In [ ]:
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

print("Selected:", audio_file)

# YAMNet expects mono 16 kHz audio
audio, sr = librosa.load(
    audio_file,
    sr=16000,
    mono=True
)

audio = audio.astype(np.float32)

print("Sample rate:", sr)
print("Duration:", round(len(audio) / sr, 2), "seconds")

In [ ]:
scores, embeddings, spectrogram = yamnet(audio)

# Average predictions across audio frames
mean_scores = tf.reduce_mean(scores, axis=0).numpy()

# Top 10 predictions
top_indices = np.argsort(mean_scores)[::-1][:10]

results = []

for idx in top_indices:
    results.append({
        "Class": class_names[idx],
        "Confidence": round(float(mean_scores[idx]) * 100, 2)
    })

results_df = pd.DataFrame(results)

results_df

,Class,Confidence
0,Speech,88.55
1,"Outside, rural or natural",0.77
2,"Outside, urban or manmade",0.67
3,"Inside, small room",0.63
4,"Narration, monologue",0.58
5,"Inside, large room or hall",0.47
6,Speech synthesizer,0.46
7,Conversation,0.41
8,Vehicle,0.36
9,Radio,0.36


In [ ]:
scores, embeddings, spectrogram = yamnet(audio)

# Average predictions across audio frames
mean_scores = tf.reduce_mean(scores, axis=0).numpy()

# Top 10 predictions
top_indices = np.argsort(mean_scores)[::-1][:10]

results = []

for idx in top_indices:
    results.append({
        "Class": class_names[idx],
        "Confidence": round(float(mean_scores[idx]) * 100, 2)
    })

results_df = pd.DataFrame(results)

results_df

,Class,Confidence
0,Speech,88.55
1,"Outside, rural or natural",0.77
2,"Outside, urban or manmade",0.67
3,"Inside, small room",0.63
4,"Narration, monologue",0.58
5,"Inside, large room or hall",0.47
6,Speech synthesizer,0.46
7,Conversation,0.41
8,Vehicle,0.36
9,Radio,0.36


In [ ]:
uploaded = files.upload()

for audio_file in uploaded.keys():

    audio, sr = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    audio = audio.astype(np.float32)

    scores, _, _ = yamnet(audio)

    mean_scores = tf.reduce_mean(scores, axis=0).numpy()

    top_indices = np.argsort(mean_scores)[::-1][:5]

    print("\n" + "=" * 60)
    print("FILE:", audio_file)
    print("=" * 60)

    for rank, idx in enumerate(top_indices, 1):
        print(
            f"{rank}. {class_names[idx]}"
            f" --> {mean_scores[idx] * 100:.2f}%"
        )